# Découvrir un LLM avec Colab

Aujourd'hui, on fait tourner un petit modèle de langage nous-mêmes.

Lance les cellules dans l'ordre avec le bouton lecture.

## 1. Préparation

Avant de commencer: dans Colab, choisis **Exécution > Modifier le type d'exécution > GPU**.

Modèle utilisé: [`Qwen/Qwen3-1.7B`](https://huggingface.co/Qwen/Qwen3-1.7B), un petit modèle disponible sur Hugging Face.

In [ ]:
# On installe les outils nécessaires.
!pip -q install -U "transformers>=4.51.0" accelerate

In [ ]:
import html
import re

import torch
from IPython.display import HTML, Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

nom_modele = "Qwen/Qwen3-1.7B"
set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("Active un GPU dans Colab: Exécution > Modifier le type d'exécution > GPU.")

tokenizer = AutoTokenizer.from_pretrained(nom_modele, trust_remote_code=True)
ids_interdits = tokenizer(["<think>", "</think>"], add_special_tokens=False).input_ids
modele = AutoModelForCausalLM.from_pretrained(
    nom_modele,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
modele.eval()

display(Markdown(f"Modèle chargé: `{nom_modele}`. Prêt à générer du texte."))

In [ ]:
def rendre_token_lisible(id_token):
    """Transforme un token en petit texte lisible."""
    texte = tokenizer.decode([int(id_token)], skip_special_tokens=False)
    if texte == "":
        texte = "∅"
    texte = texte.replace(" ", "␠").replace("\n", "↵")
    return html.escape(texte)


def nettoyer_reponse(texte):
    """Garde seulement la réponse utile du modèle."""
    if "</think>" in texte:
        texte = texte.split("</think>", 1)[1]
    texte = re.sub(r"<think>.*", "", texte, flags=re.S)
    texte = re.sub(r"<think>.*?</think>", "", texte, flags=re.S)
    for marque in ["\nUser:", "\nSystem:", "\nUtilisateur:", "\nSystème:"]:
        if marque in texte:
            texte = texte.split(marque, 1)[0]
    for debut in ["Bot:", "Assistant:", "Réponse:"]:
        if texte.strip().startswith(debut):
            texte = texte.strip()[len(debut):]
    return texte.strip()


def afficher_phrase(texte):
    display(HTML(
        "<div style='font-family:system-ui,sans-serif;font-size:20px;line-height:1.45;"
        "background:#ffffff;border:1px solid #d7dde8;border-radius:8px;padding:14px 16px;'>"
        + html.escape(texte)
        + "</div>"
    ))

## 2. Le modèle complète une phrase

Change la phrase si tu veux, puis lance la cellule suivante.

In [ ]:
texte_depart = "Le futur de l'intelligence artificielle"

In [ ]:
@torch.no_grad()
def generer_avec_details(texte_depart, nombre_tokens=20):
    entrees = tokenizer(texte_depart, return_tensors="pt").to(modele.device)
    resultat = modele.generate(
        **entrees,
        max_new_tokens=nombre_tokens,
        min_new_tokens=nombre_tokens,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    ids_generes = resultat.sequences[0, entrees.input_ids.shape[1]:]
    texte_final = tokenizer.decode(resultat.sequences[0], skip_special_tokens=True)

    cartes = []
    for numero, (id_choisi, scores) in enumerate(zip(ids_generes, resultat.scores), start=1):
        probabilites = torch.softmax(scores[0].float(), dim=-1)
        top_probs, top_ids = torch.topk(probabilites, 10)
        reste = max(0.0, 1.0 - float(top_probs.sum())) * 100

        lignes = []
        for rang, (id_token, proba) in enumerate(zip(top_ids.tolist(), top_probs.tolist()), start=1):
            pourcentage = proba * 100
            est_choisi = int(id_token) == int(id_choisi)
            classe = " option choisie" if est_choisi else " option"
            lignes.append(
                f"<div class='{classe}'>"
                f"<span class='rang'>{rang}</span>"
                f"<span class='mot'>{rendre_token_lisible(id_token)}</span>"
                f"<span class='pct'>{pourcentage:.1f}%</span>"
                f"<span class='barre'><span style='width:{max(2, min(100, pourcentage)):.1f}%'></span></span>"
                "</div>"
            )

        cartes.append(
            "<section class='carte'>"
            f"<div class='numero'>token {numero}</div>"
            f"<div class='pred'>{rendre_token_lisible(id_choisi)}</div>"
            "<div class='liste'>"
            + "".join(lignes)
            + f"<div class='autres'>+ autres: {reste:.1f}%</div>"
            + "</div></section>"
        )

    style = """
    <style>
    .zone-llm {font-family: system-ui, sans-serif; background:#f4f7f6; border:1px solid #d8e1df; border-radius:8px; padding:14px;}
    .titre-zone {font-size:18px; font-weight:700; margin:0 0 12px; color:#263238;}
    .pipeline {display:flex; gap:12px; overflow-x:auto; padding-bottom:10px;}
    .carte {min-width:220px; max-width:220px; background:#fff; border:1px solid #d8dee9; border-radius:8px; box-shadow:0 4px 12px rgba(38,50,56,.08);}
    .numero {font-size:12px; color:#64748b; padding:10px 12px 0; text-transform:uppercase; letter-spacing:.04em;}
    .pred {margin:8px 12px 10px; padding:9px 10px; min-height:24px; border-radius:6px; background:#0f766e; color:white; font-size:18px; font-weight:800; overflow-wrap:anywhere;}
    .liste {padding:0 12px 12px;}
    .option {display:grid; grid-template-columns:24px 1fr 54px; gap:6px; align-items:center; margin:6px 0; font-size:13px; color:#263238;}
    .choisie .mot, .choisie .pct {font-weight:800; color:#b45309;}
    .rang {color:#94a3b8; font-variant-numeric:tabular-nums;}
    .mot {overflow-wrap:anywhere;}
    .pct {text-align:right; font-variant-numeric:tabular-nums;}
    .barre {grid-column:2 / 4; height:5px; background:#e8edf2; border-radius:6px; overflow:hidden;}
    .barre span {display:block; height:100%; background:#f59e0b; border-radius:6px;}
    .autres {margin-top:8px; color:#475569; font-size:13px; border-top:1px solid #edf1f5; padding-top:8px;}
    </style>
    """
    html_details = style + "<div class='zone-llm'><div class='titre-zone'>Ce que le modèle hésitait à écrire</div><div class='pipeline'>" + "".join(cartes) + "</div></div>"
    return texte_final, html_details


phrase_finale, details = generer_avec_details(texte_depart, nombre_tokens=20)
afficher_phrase(phrase_finale)
display(HTML(details))

## 3. Maintenant, on lui donne un rôle

Lance **une seule** des 6 cellules suivantes pour choisir le style du bot.

In [ ]:
preprompt = """Tu es un assistant clair, sympa et patient.
Réponds en français, en 4 phrases maximum.
Ne montre pas de raisonnement caché et n'écris pas de balises techniques."""
print("Style choisi: assistant clair et sympa")

In [ ]:
preprompt = """Tu es un professeur de collège.
Explique avec des mots simples et un exemple concret.
Réponds en français, en 3 phrases maximum.
Ne montre pas de raisonnement caché et n'écris pas de balises techniques."""
print("Style choisi: professeur en 3 phrases")

In [ ]:
preprompt = """Tu es un coach motivant.
Tu encourages l'élève et tu donnes une réponse courte, utile et positive.
Réponds en français.
Ne montre pas de raisonnement caché et n'écris pas de balises techniques."""
print("Style choisi: coach motivant")

In [ ]:
preprompt = """Tu es un présentateur de quiz.
Tu réponds avec énergie, puis tu ajoutes une mini-question pour vérifier.
Réponds en français, en restant très court.
Ne montre pas de raisonnement caché et n'écris pas de balises techniques."""
print("Style choisi: présentateur de quiz")

In [ ]:
preprompt = """Tu es un scientifique enthousiaste.
Tu expliques avec une image mentale simple et un petit fait étonnant.
Réponds en français, en 4 phrases maximum.
Ne montre pas de raisonnement caché et n'écris pas de balises techniques."""
print("Style choisi: scientifique enthousiaste")

In [ ]:
preprompt = """Tu es un personnage de jeu vidéo qui donne des indices.
Tu ne donnes pas une réponse trop longue: tu aides l'élève à comprendre.
Réponds en français, avec un ton aventure.
Ne montre pas de raisonnement caché et n'écris pas de balises techniques."""
print("Style choisi: personnage de jeu vidéo")

Écris ta question ici, puis lance la cellule de réponse.

In [ ]:
question = "Pourquoi le ciel est bleu ?"

In [ ]:
@torch.no_grad()
def generer_texte(prompt, nombre_tokens=90):
    entrees = tokenizer(prompt, return_tensors="pt").to(modele.device)
    sortie = modele.generate(
        **entrees,
        max_new_tokens=nombre_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        repetition_penalty=1.08,
        bad_words_ids=ids_interdits,
        pad_token_id=tokenizer.eos_token_id,
    )
    suite = tokenizer.decode(sortie[0, entrees.input_ids.shape[1]:], skip_special_tokens=True)
    return nettoyer_reponse(suite)


if "preprompt" not in globals():
    preprompt = """Tu es un assistant clair, sympa et patient.
Réponds en français, en 4 phrases maximum.
Ne montre pas de raisonnement caché et n'écris pas de balises techniques."""

prompt_chat = f"System: {preprompt}\nUser: {question}\nBot:"
reponse = generer_texte(prompt_chat)
print(reponse)

## 4. Et si on enlève `Bot:` ?

Là, le modèle comprend moins bien qu'il doit répondre comme un bot. Il continue surtout le texte.

In [ ]:
prompt_sans_bot = f"System: {preprompt}\nUser: {question}"
suite = generer_texte(prompt_sans_bot)
print(suite)